<a href="https://colab.research.google.com/github/manuelagiraldo04/Integracion-de-datos-y-prospectiva/blob/main/Fuzzy_Logic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**Caso de estudio**
Se quiere implementar un sistema basado en los conceptos de la lógica borrosa para la gestión de pacientes en los diferentes puntos de atención que posee una EPS en Antioquia. Para esta gestión de pacientes, el punto de partida será la creación de una matriz de riesgo que permita modelar la variable de atenciones y la variable de severidad como variables linguisticas (numeros-clusters, cualidades, pertenencia). Esta matriz nos permitirá caracterizar a los pacientes en difetrnyrd niveles de riesgo, integrando información cualitativa

0. se cargan las librerías de trabajo

In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline


1. Se construye el metodo de clusterización

In [20]:
def clusterizacion(Xi):

  #Proceso de Clusterización
  XC=np.random.choice(Xi,size=5)
  XC=np.sort(XC) #Semillas de la Clusterización (Improbable, Posible, Ocasional, Probable, Frecuente)
  nc=np.zeros((len(Xi),1)) #Aquí clasifico los datos en cada una de las semillas

  for k in range(len(Xi)):
    d=np.abs(XC-Xi[k])  #Distancia de un dato a cada semilla
    nc[k]=np.argmin(d)
    nc2=np.int32(nc[k]) #Número de pertenencia al cluster en enteros
    XC[nc2,]=(XC[nc2,]+Xi[k])/2  #Actualizamos la semilla a la que pertenece cada dato (K-Medoids)

  return XC,nc   #Me retorna clusters y cluster al que pertenece un dato

2. Se procede con la clusterización de las variables de frecuencia (Atención / Año) - Severidad (Costo promedio - Atención (USD)).

In [27]:
nxl = '/content/5. Diabetes Árbol_Int_Mult.xlsx'
XDB = pd.read_excel(nxl)
XDB = XDB.dropna()    #Elimina los datos faltantes


Xf = np.array(XDB.iloc[:,9])  #Número de atenciones
Xs = np.array(XDB.iloc[:,10]) #Costo por atención

#1. Buscamos los Fuzzy Sets Xf
XCf,ncf = clusterizacion(Xf)
lbf=['Raro' , 'Poco prbable', 'Moderado', 'Probable', 'Seguro']
XCs,ncs=clusterizacion(Xs)
lbs = ['Insignificante', 'Menor', 'Significativo', 'Mayor', 'Severo']


sigmaf = []; sigmas=[] #Bases para la variable

for i in range (5):
 sigmaf.append(np.sqrt((np.sum((XCf-XCf[i,])**2)/4)))
 sigmas.append(np.sqrt((np.sum((XCs-XCs[i,])**2)/4)))

print("Los clusters de frecuencia son", XCf)
print("Los clusters de Costo-Atención (Severidad) son:", XCs)

3. Calculamos la pertenencia para un paciente que ha ido al medico durante el año 4 veces y el costo por atención promedio ha sido de 250USD

In [21]:
#(este codigo puede replicarse para todos los pacientes)

Xfp = 4 ; Xsp = 250
